# BiGRU


In [ ]:
# =========================================
# BiGRU on Mol2Vec
# =========================================
import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

# =========================================
# 0. SEED
# =========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================================
# 1. Load Mol2Vec CSV
# =========================================
train_df = pd.read_csv("/kaggle/input/mapms-alt-type1/TPCESM_Train.csv")
test_df = pd.read_csv("/kaggle/input/mapms-alt-type1/TPCESM_Test.csv")
# สมมติ column แรกเป็น ID, column สุดท้ายเป็น label
X_train = train_df.iloc[:, 1:].values.astype(np.float32)
X_test  = test_df.iloc[:, 1:].values.astype(np.float32)




# =========================================
# 2. Dataset & DataLoader (augmentation)
# =========================================
def random_dropout(seq, drop_prob=0.1):
    """Randomly drop features but keep shape."""
    mask = np.random.rand(*seq.shape) >= drop_prob
    return seq * mask

class Mol2VecSimCSEDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        x1 = random_dropout(x, 0.10)
        x2 = random_dropout(x, 0.10)
        return torch.tensor(x1, dtype=torch.float32), torch.tensor(x2, dtype=torch.float32)

train_dataset = Mol2VecSimCSEDataset(X_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0,
                          worker_init_fn=lambda worker_id: np.random.seed(SEED))

# =========================================
# 3. BiGRU model
# =========================================
class Mol2VecBiGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.bigru = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, x):
        # x: (B, D) -> (B, seq_len=1, D)
        x = x.unsqueeze(1)
        out, _ = self.bigru(x)
        # out: (B, seq_len=1, 2*hidden_dim)
        pooled = out.squeeze(1)
        return pooled

device = torch.device("cpu")
model = Mol2VecBiGRU(input_dim=X_train.shape[1]).to(device)

# =========================================
# 4. SimCSE contrastive loss
# =========================================
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
temperature = 0.05

def contrastive_loss(z1, z2, temperature):
    batch_size = z1.size(0)
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    sim_matrix = torch.matmul(z1, z2.T) / temperature
    labels = torch.arange(batch_size).long().to(device)
    return F.cross_entropy(sim_matrix, labels)

# =========================================
# 5. Training
# =========================================
EPOCHS = 100
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x1, x2 in train_loader:
        x1 = x1.to(device)
        x2 = x2.to(device)
        optimizer.zero_grad()
        z1 = model(x1)
        z2 = model(x2)
        loss = contrastive_loss(z1, z2, temperature)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.4f}")

# =========================================
# 6. Generate embeddings
# =========================================
model.eval()
with torch.no_grad():
    train_emb = model(torch.tensor(X_train, dtype=torch.float32).to(device)).cpu().numpy()
    test_emb = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

pd.DataFrame(train_emb).to_csv("BiGRU-main_train.csv", index=True)
pd.DataFrame(test_emb).to_csv("BiGRU-main_test.csv", index=True)

print("Embeddings saved.")
